# 02 · lumen against Geant4

Is lumen's light the light Geant4 makes? We check it two ways on SBND:

1. **The same event.** Notebook 00's muon and electron: Geant4 tracked their light itself and recorded every photon
   that entered a sensor (a *hit*). We send the same Geant4 steps through lumen and compare, sensor by sensor, the
   number of hits and when they arrive.
2. **The family's charge check.** Geant4 fired flashes of 128 nm light (argon's scintillation colour) from a few
   points; `lumen.compare.charge` fires the same flashes in lumen and compares every sensor. These reference runs
   are listed, with their provenance, in lumen-tpc's manifest (`lumen_tpc.REFERENCES`).

Reading the numbers: a **ratio** of totals lumen / reference with its error and **z** = (ratio - 1) / error
(|z| <= 3 passes); a **chi2/ndf** over sensors, about 1 when the sensors differ only by chance.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown

import lumen
import lumen_tpc
from lumen.compare import charge, event, references, report
from lumen.tools import plot
from lumen_tpc import runs

WORK = Path(os.environ.get("LUMEN_NOTEBOOKS", "lumen-notebooks"))
WORK.mkdir(parents=True, exist_ok=True)
path = WORK / "sbnd.lumen"
if not path.exists():
    lumen.convert("lumen_tpc:sbnd", out=path, data=references.store())
det = lumen.load(path)
events = references.product("g4ref-events-tpc-933e631d", extra=[lumen_tpc.REFERENCES])

## Two ways to run: physical, and as the reference ran

lumen's detector file holds the physics: argon's full scintillation yield, 24,000 photons per MeV, and Cherenkov
light. That is the **physical** run, the default. LArSoft, SBND's own software, multiplies the yield by a
*prescale* before Geant4 sees it (to save time) and divides it out of the sensors' efficiency later; SBND's GDML
carries that prescaled yield, 720 per MeV, and so did the Geant4 run that made our events. To compare with that
output, lumen runs with the reference's settings (`runs.config("sbnd", "gdml")`): a factor on the yield, for this run
only. The detector file is never changed.

In [ ]:
for mode in ("physical", "gdml"):
    run = runs.config("sbnd", mode)
    p = run.params(det.params())
    print(f"{mode:9s}: lights {run.lights}, LAr yield {float(p['materials/LAr/SCINTILLATIONYIELD']):.0f} /MeV  "
          f"{run.source[:90]}")

## 1. The same event, through lumen and through Geant4

`event.compare` runs the steps through lumen and returns, per sensor, lumen's expected hits and Geant4's hits, and
the arrival times of every hit. The physical run makes far more light than Geant4 did (its scintillation yield is
1 / 0.03 = 33 times higher), as it should; the run with the GDML's settings matches Geant4.

In [ ]:
muon = event.read(events / "sbnd_mu800.npz")
res = {}
for mode in ("physical", "gdml"):
    run = runs.config("sbnd", mode)
    res[mode] = event.compare(det, muon, run.params(det.params()), lights=run.lights, samples=2**18, seed=0,
                              dt=4.0, nt=1000)  # fmt: skip
    s = res[mode]["summary"]
    print(f"muon, {mode:9s}: lumen / Geant4 = {s['ratio']:.4f} ± {s['err']:.4f}; "
          f"chi2/ndf {s['chi2'] / s['ndf']:.2f} over {s['ndf']} sensors")
plot.compare(res["gdml"], title="muon (GDML settings)")
plt.show()

The arrival times show argon's two scintillation components: a fast one (ns) and a slow one (about 1.5 µs, the
straight fall on the log scale). The electron, with the same settings:

In [ ]:
electron = event.read(events / "sbnd_e30.npz")
run = runs.config("sbnd", "gdml")
r = event.compare(det, electron, run.params(det.params()), lights=run.lights, samples=2**18, seed=0, dt=4.0, nt=1000)
s = r["summary"]
print(f"electron: lumen / Geant4 = {s['ratio']:.4f} ± {s['err']:.4f}; chi2/ndf {s['chi2'] / s['ndf']:.2f}")
plot.compare(r, title="electron (GDML settings)")
plt.show()

Side by side, the muon: Geant4's hits (one event) and lumen's expectation.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
top = max(res["gdml"]["geant4"].max(), res["gdml"]["lumen"].max())
plot.display(det, res["gdml"]["geant4"], a, title="Geant4 hits (one event)", label="hits", log=True, vmax=top)
plot.display(det, res["gdml"]["lumen"], b, title="lumen (expected hits)", label="hits", log=True, vmax=top)
plt.show()

## 2. The family's charge check against Geant4

Six flashes of 128 nm light in SBND (Geant4's run on lumen's overlap-free export of the GDML, notebook 01, kept in
the reference store). `photons_factor` sets lumen's photons as a share of Geant4's; we use a tenth to keep this
quick, so lumen's own noise is part of the errors. Band 0 is the light still at 128 nm when it arrives, band 1 the
light the cathode's TPB shifted to visible wavelengths.

In [ ]:
rows = charge.check("sbnd", detector=path, extra=[lumen_tpc.REFERENCES], photons_factor=0.1)
Markdown(report.table(rows))

**Next:** 03 explains derivatives of a simulation: what they are and how to trust them.